In [53]:
###################################################
# Rating Products
###################################################

# - Average
# - Time-Based Weighted Average
# - User-Based Weighted Average
# - Weighted Rating

In [54]:

############################################
# Uygulama: Kullanıcı ve Zaman Ağırlıklı Kurs Puanı Hesaplama
############################################


In [55]:
import pandas as pd
import math
import scipy.stats as st
from sklearn.preprocessing import MinMaxScaler

In [56]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', lambda x: '%.5f' % x)

In [57]:
# (50+ Saat) Python A-Z™: Veri Bilimi ve Machine Learning
# Puan: 4.8 (4.764925)
# Toplam Puan: 4611
# Puan Yüzdeleri: 75, 20, 4, 1, <1
# Yaklaşık Sayısal Karşılıkları: 3458, 922, 184, 46, 6

In [58]:
df = pd.read_csv("course_reviews.csv")

In [59]:
df.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered
0,5.00000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.00000,0.00000,0.00000
1,5.00000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.00000,0.00000,0.00000
2,4.50000,2021-02-04 20:34:03,2019-07-04 23:23:27,1.00000,0.00000,0.00000
3,5.00000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.00000,0.00000,0.00000
4,4.00000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.00000,0.00000,0.00000


In [92]:
df.shape 
# kaç tane değerlendirme vardır?

(4323, 7)

In [93]:
print(df)


      Rating           Timestamp             Enrolled  Progress  Questions Asked  Questions Answered  days
0    5.00000 2021-02-05 07:45:55  2021-01-25 15:12:08   5.00000          0.00000             0.00000     4
1    5.00000 2021-02-04 21:05:32  2021-02-04 20:43:40   1.00000          0.00000             0.00000     5
2    4.50000 2021-02-04 20:34:03  2019-07-04 23:23:27   1.00000          0.00000             0.00000     5
3    5.00000 2021-02-04 16:56:28  2021-02-04 14:41:29  10.00000          0.00000             0.00000     5
4    4.00000 2021-02-04 15:00:24  2020-10-13 03:10:07  10.00000          0.00000             0.00000     5
5    4.00000 2021-02-04 12:42:36  2021-02-01 15:40:13   1.00000          0.00000             0.00000     5
6    5.00000 2021-02-04 12:25:30  2020-11-30 19:23:54  85.00000          0.00000             4.00000     5
7    4.50000 2021-02-04 11:13:15  2021-01-08 08:05:42  10.00000          0.00000             0.00000     5
8    5.00000 2021-02-04 08:59:53  202

In [62]:
# rating dagılımı
df["Rating"].value_counts() #Puanların dağılım bilgisine erişmek için

5.00000    3267
4.50000     475
4.00000     383
3.50000      96
3.00000      62
1.00000      15
2.00000      12
2.50000      11
1.50000       2
Name: Rating, dtype: int64

In [94]:
# Summary statistics
summary_stats = df.describe()
print("\nSummary Statistics:\n", summary_stats)


Summary Statistics:
           Rating   Progress  Questions Asked  Questions Answered       days
count 4323.00000 4323.00000       4323.00000          4323.00000 4323.00000
mean     4.76428   27.19755          0.22184             0.33102  291.15961
std      0.51959   29.14214          0.98956             6.21055  177.20125
min      1.00000    0.00000          0.00000             0.00000    4.00000
25%      5.00000    5.00000          0.00000             0.00000  140.00000
50%      5.00000   15.00000          0.00000             0.00000  286.00000
75%      5.00000   40.00000          0.00000             0.00000  411.00000
max      5.00000  100.00000         22.00000           356.00000  635.00000


In [63]:
df["Questions Asked"].value_counts() #sorulan sorular ilgili bir bilgi alalım

0.00000     3867
1.00000      276
2.00000       80
3.00000       43
4.00000       15
5.00000       13
6.00000        9
8.00000        5
9.00000        3
14.00000       2
11.00000       2
7.00000        2
10.00000       2
15.00000       2
22.00000       1
12.00000       1
Name: Questions Asked, dtype: int64

In [95]:
# Average Rating
average_rating = df['Rating'].mean()
print("\nAverage Rating:", average_rating)


Average Rating: 4.764284061993986


In [64]:
df.groupby("Questions Asked").agg({"Questions Asked": "count",
                                   "Rating": "mean"})

,Questions Asked,Rating
Questions Asked,,
0.00000,3867,4.76519
1.00000,276,4.74094
2.00000,80,4.80625
3.00000,43,4.74419
4.00000,15,4.83333
5.00000,13,4.65385
6.00000,9,5.00000
7.00000,2,4.75000
8.00000,5,4.90000


In [96]:
# Latest Timestamp
latest_timestamp = df['Timestamp'].max()
print("\nLatest Timestamp:", latest_timestamp)



Latest Timestamp: 2021-02-05 07:45:55


In [65]:
df.head()
#asıl amacımız: bu kursa verilen puanların puanını hesaplamak

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered
0,5.00000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.00000,0.00000,0.00000
1,5.00000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.00000,0.00000,0.00000
2,4.50000,2021-02-04 20:34:03,2019-07-04 23:23:27,1.00000,0.00000,0.00000
3,5.00000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.00000,0.00000,0.00000
4,4.00000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.00000,0.00000,0.00000


In [97]:
# Total Enrolled
total_enrolled = df['Enrolled'].sum()
print("\nTotal Enrolled:", total_enrolled)


Total Enrolled: 2021-01-25 15:12:082021-02-04 20:43:402019-07-04 23:23:272021-02-04 14:41:292020-10-13 03:10:072021-02-01 15:40:132020-11-30 19:23:542021-01-08 08:05:422021-02-02 18:14:492020-08-01 22:30:422021-02-01 09:54:492021-02-01 18:45:002021-02-03 20:00:042021-02-03 18:42:212020-03-29 09:07:412020-03-19 19:28:432021-02-01 09:23:312021-01-28 20:46:122019-07-25 08:04:262021-02-02 07:15:552021-02-02 17:47:412021-01-19 22:29:492021-01-03 19:42:392021-01-27 13:11:182021-02-02 16:10:522020-04-04 22:14:152021-02-02 09:07:032020-09-20 12:26:302021-02-02 10:07:222020-08-20 19:14:012021-01-31 14:58:492021-01-24 17:58:232021-02-01 12:08:222021-02-01 11:14:172021-02-01 06:15:182021-01-29 08:29:312020-11-22 16:09:362021-01-30 10:23:032021-01-31 21:29:472019-11-18 11:37:062020-12-21 16:29:262021-01-07 23:14:522021-01-13 09:37:262021-01-25 10:17:432021-01-06 10:56:432021-01-30 12:58:112019-09-26 09:11:422021-01-31 11:01:522021-01-18 20:24:402020-04-05 03:00:252020-06-19 09:33:062020-12-20 14:

In [66]:
####################
# Average
####################

In [98]:
# Average Progress
average_progress = df['Progress'].mean()
print("\nAverage Progress:", average_progress)


Average Progress: 27.197547999074718


In [99]:
# Total Questions Asked and Answered
total_questions_asked = df['Questions Asked'].sum()
total_questions_answered = df['Questions Answered'].sum()
print("\nTotal Questions Asked:", total_questions_asked)
print("Total Questions Answered:", total_questions_answered)


Total Questions Asked: 959.0
Total Questions Answered: 1431.0


In [100]:
# Correlation between columns
correlation_matrix = df.corr()
print("\nCorrelation Matrix:\n", correlation_matrix)


Correlation Matrix:
                      Rating  Progress  Questions Asked  Questions Answered     days
Rating              1.00000   0.10202         -0.01055             0.00884  0.00418
Progress            0.10202   1.00000          0.22119             0.08859 -0.07180
Questions Asked    -0.01055   0.22119          1.00000             0.14493  0.05663
Questions Answered  0.00884   0.08859          0.14493             1.00000  0.01927
days                0.00418  -0.07180          0.05663             0.01927  1.00000


In [67]:
# Ortalama Puan
df["Rating"].mean()

4.764284061993986

In [68]:
####################
# Time-Based Weighted Average
####################
# Puan Zamanlarına Göre Ağırlıklı Ortalama

In [69]:
df.head()


,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered
0,5.00000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.00000,0.00000,0.00000
1,5.00000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.00000,0.00000,0.00000
2,4.50000,2021-02-04 20:34:03,2019-07-04 23:23:27,1.00000,0.00000,0.00000
3,5.00000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.00000,0.00000,0.00000
4,4.00000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.00000,0.00000,0.00000


In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4323 entries, 0 to 4322
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Rating              4323 non-null   float64
 1   Timestamp           4323 non-null   object 
 2   Enrolled            4323 non-null   object 
 3   Progress            4323 non-null   float64
 4   Questions Asked     4323 non-null   float64
 5   Questions Answered  4323 non-null   float64
dtypes: float64(4), object(2)
memory usage: 202.8+ KB


In [103]:
# "Timestamp" sütununu datetime türüne dönüştür
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Değişiklikleri doğrulamak için DataFrame'i görüntüleyin
print(" 'Timestamp' sütununu datetime türüne dönüştürdükten sonra DataFrame:")
print(df)

 'Timestamp' sütununu datetime türüne dönüştürdükten sonra DataFrame:
      Rating           Timestamp             Enrolled  Progress  Questions Asked  Questions Answered  days
0    5.00000 2021-02-05 07:45:55  2021-01-25 15:12:08   5.00000          0.00000             0.00000     4
1    5.00000 2021-02-04 21:05:32  2021-02-04 20:43:40   1.00000          0.00000             0.00000     5
2    4.50000 2021-02-04 20:34:03  2019-07-04 23:23:27   1.00000          0.00000             0.00000     5
3    5.00000 2021-02-04 16:56:28  2021-02-04 14:41:29  10.00000          0.00000             0.00000     5
4    4.00000 2021-02-04 15:00:24  2020-10-13 03:10:07  10.00000          0.00000             0.00000     5
5    4.00000 2021-02-04 12:42:36  2021-02-01 15:40:13   1.00000          0.00000             0.00000     5
6    5.00000 2021-02-04 12:25:30  2020-11-30 19:23:54  85.00000          0.00000             4.00000     5
7    4.50000 2021-02-04 11:13:15  2021-01-08 08:05:42  10.00000          0

In [104]:

# Bugünün tarihini belirleyin
current_date = pd.to_datetime('2021-02-10 0:0:0')

# Tüm yorumların tarihlerinden bugünün tarihini çıkararak gün cinsinden ifade edin
df["days"] = (current_date - df["Timestamp"]).dt.days

# Yeni 'days' adında bir sütun oluşturduk, bu sütunda her yorumun bugüne kadar kaç gün önce yapıldığını gösterir

# DataFrame'i görüntüleyerek değişiklikleri doğrulayın
print("DataFrame after calculating 'days' since comments were made:")
print(df)


DataFrame after calculating 'days' since comments were made:
      Rating           Timestamp             Enrolled  Progress  Questions Asked  Questions Answered  days
0    5.00000 2021-02-05 07:45:55  2021-01-25 15:12:08   5.00000          0.00000             0.00000     4
1    5.00000 2021-02-04 21:05:32  2021-02-04 20:43:40   1.00000          0.00000             0.00000     5
2    4.50000 2021-02-04 20:34:03  2019-07-04 23:23:27   1.00000          0.00000             0.00000     5
3    5.00000 2021-02-04 16:56:28  2021-02-04 14:41:29  10.00000          0.00000             0.00000     5
4    4.00000 2021-02-04 15:00:24  2020-10-13 03:10:07  10.00000          0.00000             0.00000     5
5    4.00000 2021-02-04 12:42:36  2021-02-01 15:40:13   1.00000          0.00000             0.00000     5
6    5.00000 2021-02-04 12:25:30  2020-11-30 19:23:54  85.00000          0.00000             4.00000     5
7    4.50000 2021-02-04 11:13:15  2021-01-08 08:05:42  10.00000          0.00000   

In [105]:
# "days" sütununda değeri 30 günden küçük veya eşit olan satırları filtreleyerek, bu satırların "Rating" sütununun ortalamasını alır.
df.loc[df["days"] <= 30, "Rating"].mean()

4.775773195876289

In [106]:
df[df["days"] <= 30].count()

Rating                194
Timestamp             194
Enrolled              194
Progress              194
Questions Asked       194
Questions Answered    194
days                  194
dtype: int64

In [107]:
df.loc[df["days"] > 30, "Rating"] #son 30 gündeki Rate'ler geldi
df.loc[df["days"] > 30, "Rating"] .mean()

4.763744248001937

In [108]:
#30 günden fazla ve 90 günden az olan günlere bakıyoruz
df.loc[(df["days"] > 30) & (df["days"] <= 90), "Rating"].mean()


4.763833992094861

In [77]:
#eski tarihli günlere bamak istersek eğer:
df.loc[(df["days"] > 90) & (df["days"] <= 180), "Rating"].mean()

4.752503576537912

In [109]:
#demekki son zamanlarda kursun memnuniyeti ile ilgili bir artış var.
df.loc[(df["days"] > 180), "Rating"].mean()

4.76641586867305

In [110]:
# Son 30 günde yapılan yorumların ortalama puanını alarak %28'ini hesaplar.
# 30-90 gün arasındaki yorumların ortalama puanını alarak %26'sını hesaplar.
# 90-180 gün arasındaki yorumların ortalama puanını alarak %24'ünü hesaplar.
# 180 günden daha eski yorumların ortalama puanını alarak %22'sini hesaplar.
# Bu değerlerin toplamı 100'e eşit olmalıdır.
rate = (
    df.loc[df["days"] <= 30, "Rating"].mean() * 28/100 +
    df.loc[(df["days"] > 30) & (df["days"] <= 90), "Rating"].mean() * 26/100 +
    df.loc[(df["days"] > 90) & (df["days"] <= 180), "Rating"].mean() * 24/100 +
    df.loc[df["days"] > 180, "Rating"].mean() * 22/100
)

# Ağırlıklı ortalama puanı 'rate' değişkenine atar
rate


4.765025682267194

In [111]:
def time_based_weighted_average(dataframe, w1=28, w2=26, w3=24, w4=22):
    """
    Verilen DataFrame'deki yorumların tarihlerine dayalı olarak ağırlıklandırılmış bir ortalama puanı hesaplar.

    Parametreler:
    - dataframe: Değerlendirme verilerinin bulunduğu DataFrame
    - w1: Son 30 gün için kullanılacak ağırlık (varsayılan: 28)
    - w2: 30-90 gün arası için kullanılacak ağırlık (varsayılan: 26)
    - w3: 90-180 gün arası için kullanılacak ağırlık (varsayılan: 24)
    - w4: 180 günden daha eski yorumlar için kullanılacak ağırlık (varsayılan: 22)

    Dönüş Değeri:
    - Ağırlıklandırılmış ortalama puan
    """
    return dataframe.loc[dataframe["days"] <= 30, "Rating"].mean() * w1 / 100 + \
           dataframe.loc[(dataframe["days"] > 30) & (dataframe["days"] <= 90), "Rating"].mean() * w2 / 100 + \
           dataframe.loc[(dataframe["days"] > 90) & (dataframe["days"] <= 180), "Rating"].mean() * w3 / 100 + \
           dataframe.loc[(dataframe["days"] > 180), "Rating"].mean() * w4 / 100


In [112]:
time_based_weighted_average(df)

4.765025682267194

In [113]:
time_based_weighted_average(df, 30, 26, 22, 22)

4.765491074653962

In [114]:
def uniform_weighted_average(dataframe, weight=25):
    """
    Tüm yorumlar için aynı ağırlığı kullanarak bir ortalama puan hesaplar.

    Parametreler:
    - dataframe: Değerlendirme verilerinin bulunduğu DataFrame
    - weight: Tüm yorumlar için kullanılacak ağırlık (varsayılan: 25)

    Dönüş Değeri:
    - Ağırlıklandırılmış ortalama puan
    """
    return dataframe["Rating"].mean() * weight / 100


In [115]:
####################
# User-Based Weighted Average (kullanıcı temelli ağırlıklı ortalama)
####################

#user based=user quality

#acaba kursu izleme oranlarına göre daha farklı bir ağırlık mı olmalı?
df.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,days
0,5.00000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.00000,0.00000,0.00000,4
1,5.00000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.00000,0.00000,0.00000,5
2,4.50000,2021-02-04 20:34:03,2019-07-04 23:23:27,1.00000,0.00000,0.00000,5
3,5.00000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.00000,0.00000,0.00000,5
4,4.00000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.00000,0.00000,0.00000,5


In [116]:
# Gruplama yaparak her bir ilerleme seviyesi için ortalama puanı hesaplayın
df.groupby("Progress").agg({"Rating": "mean"})

# İlerleme seviyesine göre ağırlıklandırılmış ortalama puanı hesaplayın
df.loc[df["Progress"] <= 10, "Rating"].mean() * 22 / 100 + \
    df.loc[(df["Progress"] > 10) & (df["Progress"] <= 45), "Rating"].mean() * 24 / 100 + \
    df.loc[(df["Progress"] > 45) & (df["Progress"] <= 75), "Rating"].mean() * 26 / 100 + \
    df.loc[df["Progress"] > 75, "Rating"].mean() * 28 / 100


4.800257704672543

In [117]:
def user_based_weighted_average(dataframe, w1=22, w2=24, w3=26, w4=28):
    """
    Her bir kullanıcının ilerleme seviyesine göre ağırlıklandırılmış bir ortalama puan hesaplar.

    Parametreler:
    - dataframe: Kullanıcıların ilerleme ve puanlama verilerinin bulunduğu DataFrame
    - w1: 10 veya daha az ilerleme seviyesi için kullanılacak ağırlık (varsayılan: 22)
    - w2: 10-45 arası ilerleme seviyesi için kullanılacak ağırlık (varsayılan: 24)
    - w3: 45-75 arası ilerleme seviyesi için kullanılacak ağırlık (varsayılan: 26)
    - w4: 75'ten daha fazla ilerleme seviyesi için kullanılacak ağırlık (varsayılan: 28)

    Dönüş Değeri:
    - Ağırlıklandırılmış ortalama puan
    """
    return dataframe.loc[dataframe["Progress"] <= 10, "Rating"].mean() * w1 / 100 + \
           dataframe.loc[(dataframe["Progress"] > 10) & (dataframe["Progress"] <= 45), "Rating"].mean() * w2 / 100 + \
           dataframe.loc[(dataframe["Progress"] > 45) & (dataframe["Progress"] <= 75), "Rating"].mean() * w3 / 100 + \
           dataframe.loc[dataframe["Progress"] > 75, "Rating"].mean() * w4 / 100


In [118]:
user_based_weighted_average(df, 20, 24, 26, 30)

4.803286469062915

In [119]:
####################
# Weighted Rating (time based + user base) bir araya getirilerek ağırlıklandırma çalışması
####################

In [120]:
def course_weighted_rating(dataframe, time_w=50, user_w=50):
    """
    Zaman ve kullanıcı bazında ağırlıklandırılmış bir ortalama puan hesaplar.

    Parametreler:
    - dataframe: Kullanıcıların ilerleme ve puanlama verilerinin bulunduğu DataFrame
    - time_w: Zaman bazında kullanılacak ağırlık (varsayılan: 50)
    - user_w: Kullanıcı bazında kullanılacak ağırlık (varsayılan: 50)

    Dönüş Değeri:
    - Zaman ve kullanıcı bazında ağırlıklandırılmış ortalama puan
    """
    return (time_based_weighted_average(dataframe) * time_w / 100) + \
           (user_based_weighted_average(dataframe) * user_w / 100)


In [121]:
course_weighted_rating(df)

4.782641693469868

In [122]:
course_weighted_rating(df, time_w=40, user_w=60)

4.786164895710403